# TRAILS: Simple Numerical Example (Tutorial)

**Authors**: Romain Sacchi (PSI)

**Contact**: romain.sacchi@psi.ch

## Purpose

This notebook is a guided, end-to-end first experience with TRAILS. It covers:

1. Loading a small example data package.
2. Exploring the technosphere (`A`) and biosphere (`B`) matrices.
3. Temporal routing and graph visualization.
4. Running temporal and static LCA, then plotting results.
5. Linking results to the FaIR climate emulator (radiative forcing + temperature).
## Theory (short)

TRAILS (Temporal Routing And Aggregation of Impacts across Life-cycle Systems)
extends classic LCA by making time an explicit dimension. Exchanges can carry
**temporal distributions** (discrete, normal, lognormal, uniform, triangular),
which are expanded into year offsets during traversal. For each calendar year
that becomes active, the system is solved and biosphere flows are accumulated
at their respective years. This yields **time-resolved inventories** and **impact
scores** that answer questions like *when impacts occur* and *which upstream
suppliers dominate over time*.

**Environment requirements (from `pyproject.toml`):**
- Python `>=3.10, <3.13`
- Dependencies are loaded from `requirements.txt`
- Optional extras: `testing` (pytest), `docs` (sphinx)


In [ ]:
# Core imports
from pathlib import Path
from datapackage import Package

from trails import (
    Trails,
    get_lcia_method_names,
    plot_temporal_scores,
    plot_rf,
    plot_temp,
    clear_cache,
)


## 1. Load the example data package

We use the small data package shipped in `examples/example data package`.


**Caching note**: the first time you load a data package, TRAILS builds
        and caches the technosphere and biosphere matrices (and their indices).
        Subsequent runs reuse the cache for faster startup. If you change the
        data package or want a clean rebuild, use `clear_cache()` to remove the
        cached matrices and force regeneration on the next load.


In [ ]:
# Point to the example data package
package_path = Path('examples/example data package/datapackage.json')
package = Package(str(package_path))

# Initialize Trails
trails = Trails(package)


In [ ]:
# Optional: clear cache to force rebuilding matrices on next load
clear_cache()


## 2. Explore matrices and indices (A, B)


TRAILS exposes the technosphere matrix `A` and biosphere matrix `B` as 3D arrays:

- `A` indexed by `(year, activity, activity)`
- `B` indexed by `(year, activity, flow)`

These are the time-resolved system matrices used during traversal. You can inspect
shapes and a few entries to get a feel for the data.


In [ ]:
# Inspect matrix shapes (year, activity, activity) and (year, activity, flow)
        print('A shape:', trails.A.shape)
        print('B shape:', trails.B.shape)


You can also search activities by name and inspect exchanges for a
        specific activity.


In [ ]:
from trails import search_activity

        # Find activity ids by keyword
        search_activity(trails, 'electricity')[:5]


In [ ]:
# Print exchanges for a chosen activity index (example: 0)
        # Replace 0 with an id from search_activity if desired.
        trails.print_exchange_table(0)


## 3. Choose an LCIA method

TRAILS exposes a helper that lists all available LCIA methods in the package.


In [ ]:
methods = get_lcia_method_names(trails)
methods[:5]  # show a few


Pick a method and keep it in a single-element list (the LCA runner expects a list).

In [ ]:
method = methods[0]
method


## 4. Run the temporal LCA

This computes time-resolved scores and inventory. We also keep the inventory so we
can later use the climate emulator.


### 4.1 Temporal routing (graph traversal)

                This step constructs a temporal routing graph from a starting activity.
                It is useful for understanding time-dependent dependencies before solving.


In [ ]:
# Example temporal routing (graph traversal)
                trails.temporal_routing(
                    start_year=2050,
                    start_act_idx=13,
                    amount=1.0,
                    max_depth=3,
                    show_progress=True,
                )


You can visualize the routing graph with `plot_temporal_graph`.


In [ ]:
from trails.plotting import plot_temporal_graph

                plot_temporal_graph(
                    trails,
                    filename='trails_graph.html',
                    notebook=False,
                )


### 4.2 Temporal LCA (full solve)

                This computes time-resolved scores and inventory. We keep the inventory
                for the climate emulator later on.


In [ ]:
trails.lca(
                    methods=[method],
                    show_progress=True,
                    compute_score=True,
                    store_inventory=True,
                )


### 4.3 Static LCA (single year)

                This provides a quick single-year check for a chosen activity.


In [ ]:
# Example static LCA for a specific activity index
                # Replace act_idx with an index of interest
                trails.static_lca(year=2050, act_idx=13, methods=[method])
                trails.static_score


## 5. Plot temporal impacts

The default plot highlights the top contributors over time.


In [ ]:
fig = plot_temporal_scores(
    trails=trails,
    method_label=str(method),
    stacked=True,
    legend_top_n=6,
    year_range=(2000, 2100),
    show_cumulative_axis=True,
)
fig


## 6. Quick static check (optional)

Static LCA uses a single year and activity index (see `Trails.static_lca`).


In [ ]:
# Example: run a static LCA for a given year/activity index
# You can look up indices with your own metadata if needed.
# trails.static_lca(year=2050, act_idx=13, methods=[method])
# trails.static_score


## 7. Climate emulator (FaIR)

TRAILS can translate inventory time series into **radiative forcing** and
**temperature anomaly** using the FaIR climate model.
Outputs are stored as quantiles across all FaIR configurations:
`2.5, 25, 50, 75, 97.5`.


In [ ]:
from trails.fair_rf import run_fair_delta_rf

rf = run_fair_delta_rf(
    trails,
    scenario='high-extension',
    scale_target_fraction=0.1,
)

# Access stored results
trails.instant_radiative_forcing
trails.delta_temperature


### Plot radiative forcing (median + quantile band)

In [ ]:
fig_rf = plot_rf(
    trails,
    by='flow',
    year_range=(2000, 2100),
    year_tick=10,
    reference_year=2050,
)
fig_rf


### Plot temperature anomaly (median + quantile band)

In [ ]:
fig_temp = plot_temp(
    trails,
    by='root activity',
    year_range=(2000, 2100),
    year_tick=10,
)
fig_temp


## 8. Next steps

- Import user inventories (Excel) with `trails.import_excel_inventory`
- Run additional LCIA methods and compare results
- Customize plots (stacked vs non-stacked, flow grouping, etc.)
